

# IMPORT LIBRARIES



In [1]:
import zipfile
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    LSTM,
    TimeDistributed,
    GlobalAveragePooling2D,
    Reshape
)

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

print("Main libraries imported successfully!")

Main libraries imported successfully!


In [2]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

# EXTRACT ZIP FILE AND LOAD DATA

In [ ]:
import os

extract_path = r"C:\Users\User\Downloads\archive\Data"

classes = os.listdir(extract_path)

print("Classes:")
for cls in classes:
    print(cls)

# CHECK DATASET FOLDERS

In [ ]:
import os
print(os.listdir(data_path))

# IMAGE PREPROCESSING

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_data = train_datagen.flow_from_directory(
    extract_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_data = train_datagen.flow_from_directory(
    extract_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# DISPLAY CLASS LABELS

In [ ]:
class_names = list(train_data.class_indices.keys())

print("\nClass Labels:")
print(class_names)

# VISUALIZE MRI IMAGES

In [ ]:
images, labels = next(train_data)

# Get class names
class_names = list(train_data.class_indices.keys())

plt.figure(figsize=(12,10))

for i in range(9):

    plt.subplot(3,3,i+1)

    plt.imshow(images[i])

    # Convert one-hot encoded label to class index
    label_index = np.argmax(labels[i])

    # Get actual label name
    label_name = class_names[label_index]

    plt.title(label_name, fontsize=10)

    plt.axis("off")

plt.tight_layout()

plt.show()

# PART A — CNN MODEL

## BUILD CNN MODEL

In [ ]:
cnn_model = Sequential([

    Conv2D(32, (3,3), activation='relu',
           input_shape=(224,224,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(256, activation='relu'),
    Dropout(0.5),

    Dense(4, activation='softmax')

])

## COMPILE CNN MODEL

In [ ]:
cnn_model.compile(
    optimizer=Adam(0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

##  TRAIN CNN MODEL

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

cnn_history = cnn_model.fit(
    train_data,
    validation_data=val_data,
    epochs=5,
    callbacks=[early_stop]
)

## CNN EVALUATION

In [ ]:
cnn_predictions = cnn_model.predict(val_data)

y_pred = np.argmax(cnn_predictions, axis=1)
y_true = val_data.classes

cnn_accuracy = accuracy_score(y_true, y_pred)

print("\nCNN Accuracy:", cnn_accuracy)

## CNN CLASSIFICATION REPORT

In [ ]:
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

## CONFUSION MATRIX (CNN)

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("CNN Confusion Matrix")

plt.show()

# PART B: ML MODEL--RANDOM FOREST

In [ ]:
import cv2
import numpy as np

IMG_SIZE = 32

X = []
y = []

classes = [
    "Mild Dementia",
    "Moderate Dementia",
    "Non Demented",
    "Very mild Dementia"
]

for label, class_name in enumerate(classes):

    class_path = os.path.join(extract_path, class_name)

    print(f"\nLoading: {class_name}")

    if not os.path.exists(class_path):

        print("Folder NOT found")
        continue

    image_files = os.listdir(class_path)

    print(f"Found {len(image_files)} images")

    for img_name in image_files:

        try:

            img_path = os.path.join(class_path, img_name)

            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

            img = img / 255.0

            img = img.flatten()

            X.append(img)

            y.append(label)

        except Exception as e:

            print("Error:", e)

X = np.array(X)

y = np.array(y)

print("\nDataset Shape:")

print("X Shape:", X.shape)

print("y Shape:", y.shape)

## SPLIT DATASET

In [ ]:
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

## TRAIN RANDOM FOREST MODEL

In [ ]:
rf_model = RandomForestClassifier(

    n_estimators=100,

    max_depth=15,

    random_state=42,

    n_jobs=-1
)

In [ ]:
print("\nTraining Random Forest Model...\n")

rf_model.fit(X_train, y_train)

print("Training Completed")

## EVALUATIONS AND PREDICTIONS

In [ ]:
y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"\nRandom Forest Accuracy: {accuracy:.4f}")

## CLASSIFICATION REPORT

In [ ]:
print("\nClassification Report:\n")

print(classification_report(
    y_test,
    y_pred,
    target_names=classes
))

## CONFUSION MATRIX

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8,6))

sns.heatmap(

    cm,

    annot=True,

    fmt='d',

    cmap='Blues',

    xticklabels=classes,

    yticklabels=classes
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("Random Forest Confusion Matrix")

plt.show()

# PART C — ARTIFICIAL NEURAL NETWORK (ANN) MODEL

## BUILD ANN MODEL

In [ ]:
ann_model = Sequential([

    Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.4),

    Dense(256, activation='relu'),
    Dropout(0.3),

    Dense(128, activation='relu'),
    Dropout(0.2),

    Dense(4, activation='softmax')

])

## COMPILE ANN MODEL

In [ ]:
ann_model.compile(
    optimizer=Adam(0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()

## TRAIN ANN MODEL

In [ ]:
early_stop_ann = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

ann_history = ann_model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=2,
    batch_size=32,
    callbacks=[early_stop_ann]
)

## ANN EVALUATION

In [ ]:
ann_predictions = ann_model.predict(X_test)

ann_y_pred = np.argmax(ann_predictions, axis=1)

ann_accuracy = accuracy_score(y_test, ann_y_pred)

print("\nANN Accuracy:", ann_accuracy)

## ANN CLASSIFICATION REPORT

In [ ]:
print("\nClassification Report:\n")

print(classification_report(
    y_test,
    ann_y_pred,
    target_names=classes
))

## CONFUSION MATRIX (ANN)

In [ ]:
cm_ann = confusion_matrix(y_test, ann_y_pred)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm_ann,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=classes,
    yticklabels=classes
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("ANN Confusion Matrix")

plt.show()

# MODEL COMPARISON

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Random Forest Metrics
rf_accuracy = accuracy_score(y_test, y_pred)

rf_precision = precision_score(
    y_test,
    y_pred,
    average='weighted'
)

rf_recall = recall_score(
    y_test,
    y_pred,
    average='weighted'
)

rf_f1 = f1_score(
    y_test,
    y_pred,
    average='weighted'
)

# CNN Metrics
cnn_y_pred = np.argmax(cnn_predictions, axis=1)

cnn_accuracy = accuracy_score(
    y_true,
    cnn_y_pred
)

cnn_precision = precision_score(
    y_true,
    cnn_y_pred,
    average='weighted'
)

cnn_recall = recall_score(
    y_true,
    cnn_y_pred,
    average='weighted'
)

cnn_f1 = f1_score(
    y_true,
    cnn_y_pred,
    average='weighted'
)

# ANN Metrics
ann_accuracy = accuracy_score(
    y_test,
    ann_y_pred
)

ann_precision = precision_score(
    y_test,
    ann_y_pred,
    average='weighted'
)

ann_recall = recall_score(
    y_test,
    ann_y_pred,
    average='weighted'
)

ann_f1 = f1_score(
    y_test,
    ann_y_pred,
    average='weighted'
)

comparison_df = pd.DataFrame({

    "Model": [
        "Random Forest",
        "CNN",
        "ANN"
    ],

    "Accuracy": [
        rf_accuracy,
        cnn_accuracy,
        ann_accuracy
    ],

    "Precision": [
        rf_precision,
        cnn_precision,
        ann_precision
    ],

    "Recall": [
        rf_recall,
        cnn_recall,
        ann_recall
    ],

    "F1-Score": [
        rf_f1,
        cnn_f1,
        ann_f1
    ]
})

comparison_df

In [ ]:
models = ['Random Forest', 'CNN', 'ANN']

accuracies = [
    rf_accuracy,
    cnn_accuracy,
    ann_accuracy
]

plt.figure(figsize=(7,5))

bars = plt.bar(models, accuracies)

plt.title("Accuracy Comparison")

plt.ylabel("Accuracy")

plt.ylim(0,1)

# Display values on bars
for bar in bars:

    yval = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width()/2,
        yval + 0.01,
        round(yval,4),
        ha='center'
    )

plt.show()

In [ ]:
comparison_df.set_index("Model").plot(

    kind='bar',

    figsize=(10,6)
)

plt.title("Random Forest vs CNN vs ANN Performance Comparison")

plt.ylabel("Score")

plt.ylim(0,1)

plt.xticks(rotation=0)

plt.grid(axis='y')

plt.show()

# PART D — GRAD-CAM EXPLAINABILITY

## SELECT TEST IMAGE

In [ ]:
img_path = val_data.filepaths[0]

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)/255.0
img_array_expanded = np.expand_dims(img_array, axis=0)

## GRAD-CAM FUNCTION

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Model

# Select last convolutional layer
last_conv_layer = cnn_model.layers[4]

# Create Grad-CAM model
grad_model = Model(
    inputs=cnn_model.inputs,
    outputs=[last_conv_layer.output, cnn_model.outputs[0]]
)

# Compute gradients
with tf.GradientTape() as tape:

    # Forward pass
    conv_outputs, predictions = grad_model(img_array_expanded)

    # Predicted class index
    predicted_class = tf.argmax(predictions[0])

    # Loss for predicted class
    loss = predictions[:, predicted_class]

# Compute gradients of loss wrt conv outputs
grads = tape.gradient(loss, conv_outputs)

# Global average pooling of gradients
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

# Remove batch dimension
conv_outputs = conv_outputs[0]

# Weight feature maps by gradients
heatmap = tf.reduce_sum(
    conv_outputs * pooled_grads,
    axis=-1
)

# ReLU activation
heatmap = tf.maximum(heatmap, 0)

# Normalize heatmap
heatmap /= tf.reduce_max(heatmap)

# Convert to numpy
heatmap = heatmap.numpy()

## DISPLAY GRAD-CAM

In [ ]:
plt.figure(figsize=(6,6))

plt.imshow(heatmap, cmap='jet')

plt.title("GRAD-CAM Heatmap")

plt.colorbar()

plt.show()

# PART E — SHAP EXPLAINABILITY

## SHAP EXPLANATION

In [ ]:
%pip install shap==0.41.0 --no-deps

In [3]:
import sys
import numpy as np
import tensorflow as tf

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)

Python executable: C:\Users\Huawei\AppData\Local\Programs\Python\Python313\python.exe
Python version: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
TensorFlow: 2.21.0
NumPy: 2.5.2


In [4]:
background = images[:20]

explainer = shap.DeepExplainer(
    cnn_model,
    background
)

shap_values = explainer.shap_values(images[:5])

shap.image_plot(shap_values, images[:5])

NameError: name 'images' is not defined

# PART F — LIME EXPLAINABILITY

## LIME EXPLAINER

In [5]:
explainer = lime_image.LimeImageExplainer()

explanation = explainer.explain_instance(

    img_array.astype('double'),

    cnn_model.predict,

    top_labels=1,

    hide_color=0,

    num_samples=1000
)

temp, mask = explanation.get_image_and_mask(

    explanation.top_labels[0],

    positive_only=True,

    num_features=10,

    hide_rest=False
)

NameError: name 'lime_image' is not defined

## DISPLAY LIME RESULT

In [6]:
plt.figure(figsize=(8,8))

plt.imshow(mark_boundaries(temp / 2 + 0.5, mask))

plt.title("LIME Explanation")

plt.axis('off')

plt.show()

NameError: name 'mark_boundaries' is not defined

<Figure size 800x800 with 0 Axes>

# PART G — PREDICTION FUNCTION

## PREDICT NEW MRI IMAGE

In [8]:
def predict_mri(img_path):

    img = image.load_img(
        img_path,
        target_size=(224,224)
    )

    img_array = image.img_to_array(img)/255.0

    img_array = np.expand_dims(img_array, axis=0)

    prediction = cnn_model.predict(img_array)

    predicted_class = class_names[np.argmax(prediction)]

    confidence = np.max(prediction)*100

    plt.figure(figsize=(6,6))

    plt.imshow(img)

    plt.axis('off')

    plt.title(
        f"Prediction: {predicted_class}\n"
        f"Confidence: {confidence:.2f}%"
    )

    plt.show()

In [9]:
predict_mri("/content/Data/Mild Dementia/OAS1_0031_MR1_mpr-2_130.jpg")

FileNotFoundError: [Errno 2] No such file or directory: '/content/Data/Mild Dementia/OAS1_0031_MR1_mpr-2_130.jpg'

# PART H — SAVE MODELS

In [10]:
cnn_model.save("alzheimers_cnn_model.h5")

ann_model.save("alzheimers_ann_model.h5")

print("\nModels Saved Successfully")

NameError: name 'cnn_model' is not defined